# Stage A2 — Exploratory Analysis

Dual-fuel PINN pipeline | Sandrine Schueller Mafra | PPGEM – UFPR

**Input:** `data/masters_data.xlsx` (read directly — this notebook has no
hard-coded values from the dissertation text, since Stage A1 already
showed several of those don't match the real data).

**What this notebook computes, straight from the data:**
- Descriptive statistics
- Per-variable distribution + kernel density estimate (KDE)
- Pearson and Spearman correlation matrices (9×9, inputs and outputs)
- Variance Inflation Factor (VIF) for the 4 input variables
- IQR-based outlier screening per variable
- Every Table 7 constraint (Eq. 3.9–3.14 and the η bounds) checked against the data — the
  gradient-based ones measured **within** their own OFAT block

**Output:** every figure and result table is saved to `outputs/html/` as
`A2_<section>[_qualifier].html`. No data files are written — no later
stage depends on this notebook.

## Setup

In [ ]:
import numpy as np
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import gaussian_kde, spearmanr
from pathlib import Path

print("polars ", pl.__version__)
import plotly
print("plotly ", plotly.__version__)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "masters_data.xlsx"
OUT_DIR = PROJECT_ROOT / "outputs"
RAW_PATH

## Color palette and output naming (shared across the whole pipeline)

`SPLIT_COLORS` (semantic role: train / validation / test / reference
value / alert / neutral) and `VARIABLE_COLORS` (identity of each of the
4 inputs and 5 outputs) are identical in every A/B/C notebook, so the
same element always has the same color in any chart of the pipeline.

Every figure or result table generated below is also saved to
`outputs/html/`, named `A2_<section>[_qualifier].html` — the number
matches the corresponding section header, so the order in which each
output was produced can be read from the file name alone.

In [ ]:
SPLIT_COLORS = {
    "train": "#B7C9DA",
    "validation": "#2B6EFF",
    "test": "#571D99",
    "reference": "#343A40",   # value transcribed from the dissertation text
    "alert": "#E85D04",       # outlier / out of range / anomaly
    "neutral": "#B0AFA8",     # grid lines / neutral reference
}
VARIABLE_COLORS = {
    "SOI": "#073b3a", "lambda": "#0b6e4f", "sub_rate": "#08a045", "P_rail": "#6bbf59",
    "NOx": "#c7adff", "PM": "#916dd5", "eta": "#7151a9", "HC": "#573d7f", "CO2": "#46325d",
}

HTML_DIR = OUT_DIR / "html"
HTML_DIR.mkdir(parents=True, exist_ok=True)


def flagged_table_html(df, title, out_path, flag_col=None, is_flagged=lambda v: False, ref_cols=()):
    """Result table -> Plotly go.Table -> HTML.
    Columns listed in ref_cols get the 'reference' tone in the header
    (values transcribed from the dissertation text). Cells in flag_col
    get the 'alert' tone wherever is_flagged(value) is True."""
    cols = list(df.columns)
    n = df.shape[0]
    header_fill = [SPLIT_COLORS["reference"] if c in ref_cols else "#F1F3F5" for c in cols]
    header_font = ["white" if c in ref_cols else "black" for c in cols]
    cell_fill = []
    for c in cols:
        if c == flag_col:
            cell_fill.append([SPLIT_COLORS["alert"] if is_flagged(v) else "white"
                               for v in df[c].to_list()])
        else:
            cell_fill.append(["white"] * n)
    fig = go.Figure(data=[go.Table(
        header=dict(values=cols, fill_color=header_fill,
                     font=dict(color=header_font), align="left"),
        cells=dict(values=[df[c].to_list() for c in cols],
                    fill_color=cell_fill, align="left"),
    )])
    fig.update_layout(title=title, margin=dict(t=40, l=10, r=10, b=10))
    fig.write_html(str(out_path), include_plotlyjs="inline")
    return fig


def simple_table_html(df, title, out_path):
    return flagged_table_html(df, title, out_path)

## 1. Load data

Reads the Excel file directly with `polars` (requires the optional
`fastexcel` dependency: `pip install polars fastexcel` if this errors
with a missing-engine message) and renames columns to short, code-
friendly names. The raw `SO_H [FSN]` column is the particulate-matter
channel and `ETA [%]` is stored as a **fraction** (0–1) in the file
despite its header — both handled explicitly below, nothing silently
converted.

In [ ]:
COLUMN_MAP = {
    "SOI [o.CA]": "SOI",
    "Lambda [-]": "lambda",
    "Sub. Rate [%]": "sub_rate",
    "Prail [bar]": "P_rail",
    "HC [g/kW.h]": "HC",
    "NOX [ppm]": "NOx",
    "CO2 [%]": "CO2",
    "SO_H [FSN]": "PM",
    "ETA [%]": "eta",
}
INPUT_COLS = ["SOI", "lambda", "sub_rate", "P_rail"]
OUTPUT_COLS = ["HC", "NOx", "CO2", "PM", "eta"]
ALL_COLS = INPUT_COLS + OUTPUT_COLS

df = pl.read_excel(RAW_PATH)
df = df.rename(COLUMN_MAP).select(ALL_COLS)

print(df.shape)
df.head()


## 2. Descriptive statistics

In [ ]:
desc = df.describe()
simple_table_html(desc, "A2 -- Descriptive statistics", HTML_DIR / "A2_02_descriptive_stats.html")
desc

## 3. Distribution & density (KDE) per variable

Histogram (density-normalised) with a Gaussian KDE overlay, one panel
per variable, computed straight from the loaded data.

In [ ]:
fig = make_subplots(rows=3, cols=3, subplot_titles=ALL_COLS)

for i, col in enumerate(ALL_COLS):
    row, c = divmod(i, 3)
    values = df[col].to_numpy()
    kde = gaussian_kde(values)
    x_grid = np.linspace(values.min(), values.max(), 200)
    density = kde(x_grid)
    color = VARIABLE_COLORS[col]

    fig.add_trace(
        go.Histogram(x=values, histnorm="probability density",
                     marker_color=color, opacity=0.45, showlegend=False,
                     nbinsx=12),
        row=row + 1, col=c + 1,
    )
    fig.add_trace(
        go.Scatter(x=x_grid, y=density, mode="lines",
                    line=dict(color=color, width=2.5), showlegend=False),
        row=row + 1, col=c + 1,
    )

fig.update_layout(height=800, width=950,
                   title_text="Distribution + KDE per variable (masters_data.xlsx)")
fig.show()
fig.write_html(str(HTML_DIR / "A2_03_kde.html"), include_plotlyjs="inline")

## 4. Correlation — Pearson

Computed with `numpy.corrcoef` on the raw column values (not polars'
own `.corr`, so the result doesn't depend on which correlation method a
given polars version ships) and displayed as an interactive heatmap.

**Read these matrices with care:** they pool all 40 rows across every
OFAT block. When only one input moves at a time, the global correlation
between, say, PM and SOI is diluted by all the rows where SOI sits at
its baseline while a different input is being swept. The sign of each
physics constraint should be read in **Section 8**, which measures each
relationship inside its own OFAT block — not here.

The diverging scale (`RdBu_r`) is kept on purpose: the sign of a
correlation is a different axis from the ones the project palette
encodes (train/val/test and variable identity), and a standard diverging
scale is the most direct way to read it.

In [ ]:
X = df.select(ALL_COLS).to_numpy()
pearson_mat = np.corrcoef(X, rowvar=False)

pearson_data = {"variable": ALL_COLS}
for i, c in enumerate(ALL_COLS):
    pearson_data[c] = np.round(pearson_mat[:, i], 3)
pearson_df = pl.DataFrame(pearson_data)
simple_table_html(pearson_df, "A2 -- Pearson correlation (all 40 rows)", HTML_DIR / "A2_04_pearson_table.html")
pearson_df

In [ ]:
fig = px.imshow(
    pearson_mat, x=ALL_COLS, y=ALL_COLS, text_auto=".2f",
    color_continuous_scale="RdBu_r", zmin=-1, zmax=1, aspect="auto",
    title="Pearson correlation matrix (all 40 rows)",
)
fig.update_layout(width=650, height=600)
fig.show()
fig.write_html(str(HTML_DIR / "A2_04_pearson_heatmap.html"), include_plotlyjs="inline")

## 5. Correlation — Spearman

Rank-based correlation (via `scipy.stats.spearmanr`); picks up monotonic
relationships that aren't necessarily linear, which matters here since
several inputs/outputs are physically nonlinear (e.g. a U-shaped HC–λ
relationship would show up weakly in Pearson but should still show up
here if it's monotonic on each side).

In [ ]:
spearman_mat, _ = spearmanr(X)
fig = px.imshow(
    spearman_mat, x=ALL_COLS, y=ALL_COLS, text_auto=".2f",
    color_continuous_scale="RdBu_r", zmin=-1, zmax=1, aspect="auto",
    title="Spearman correlation matrix (all 40 rows)",
)
fig.update_layout(width=650, height=600)
fig.show()
fig.write_html(str(HTML_DIR / "A2_05_spearman_heatmap.html"), include_plotlyjs="inline")

## 6. Multicollinearity — Variance Inflation Factor (inputs only)

VIF is computed here with plain linear algebra (no `statsmodels`
dependency): for standardised predictors, `VIF_j` is the *j*-th diagonal
element of the inverse of their correlation matrix — algebraically the
same result `1 / (1 - R²_j)` from regressing each input on the other
three would give.

Rule of thumb: VIF < 5 is comfortable, VIF > 10 signals a real problem.

In [ ]:
Xin = df.select(INPUT_COLS).to_numpy()
Xin_std = (Xin - Xin.mean(axis=0)) / Xin.std(axis=0, ddof=1)
R = np.corrcoef(Xin_std, rowvar=False)
vif = np.diag(np.linalg.inv(R))

vif_df = pl.DataFrame({"variable": INPUT_COLS, "VIF": np.round(vif, 3)})
flagged_table_html(vif_df, "A2 -- Variance Inflation Factor (inputs)", HTML_DIR / "A2_06_vif_table.html",
                    flag_col="VIF", is_flagged=lambda v: v > 5)
vif_df

In [ ]:
fig = go.Figure(go.Bar(x=INPUT_COLS, y=vif,
                        marker_color=[VARIABLE_COLORS[c] for c in INPUT_COLS]))
fig.add_hline(y=5, line_dash="dash", line_color=SPLIT_COLORS["alert"],
              annotation_text="VIF = 5 (caution threshold)")
fig.update_layout(title="Variance Inflation Factor per input", yaxis_title="VIF",
                   width=650, height=400)
fig.show()
fig.write_html(str(HTML_DIR / "A2_06_vif_bar.html"), include_plotlyjs="inline")

## 7. Outlier screening (IQR method)

Standard Tukey rule: a value is flagged if it falls outside
`[Q1 − 1.5·IQR, Q3 + 1.5·IQR]`.

**Read the counts with this dataset's design in mind:** the test matrix
is one-factor-at-a-time (Sec. 3.1.1.2), so most rows repeat a shared
baseline for three of the four inputs while only one is swept. That
means a variable's own inter-quartile range can be very narrow (many
rows sit exactly at the baseline), so genuine, intentional sweep points
get flagged as "outliers" here even though they're the most informative
rows in the dataset, not data-quality problems. Treat this table as
*"far from the modal condition"*, not *"suspect measurement."*

For the same reason, the bars below use each variable's identity color
rather than the palette's alert color: painting them as alerts would
suggest a data-quality problem that this count does not demonstrate.

In [ ]:
def iqr_outliers(frame: pl.DataFrame, col: str):
    q1 = frame[col].quantile(0.25, interpolation="linear")
    q3 = frame[col].quantile(0.75, interpolation="linear")
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    flagged = frame.filter((pl.col(col) < lo) | (pl.col(col) > hi))
    return len(flagged), lo, hi, flagged


rows = []
for c in ALL_COLS:
    n, lo, hi, _ = iqr_outliers(df, c)
    rows.append({"variable": c, "n_outliers": n,
                 "lower_bound": round(lo, 4), "upper_bound": round(hi, 4)})

outlier_summary = pl.DataFrame(rows)
simple_table_html(outlier_summary, "A2 -- IQR screening per variable", HTML_DIR / "A2_07_outlier_table.html")
outlier_summary

In [ ]:
fig = go.Figure(go.Bar(x=outlier_summary["variable"], y=outlier_summary["n_outliers"],
                        marker_color=[VARIABLE_COLORS[c] for c in outlier_summary["variable"].to_list()]))
fig.update_layout(title="IQR-flagged points per variable (see note above before reacting to this)",
                   yaxis_title="count", width=650, height=400)
fig.show()
fig.write_html(str(HTML_DIR / "A2_07_outlier_bar.html"), include_plotlyjs="inline")

## 8. Table 7 constraints checked against the data

Table 7 (Sec. 3.3) lists seven constraints. The matrices in Sections 4–5
pool every OFAT block, so they cannot be used to check the gradient-based
ones. Here each of those is measured only on the rows of the block in
which the relevant input is being swept (same block inference as A1,
Section 9, repeated here so A2 keeps running on its own). The two bounding
constraints are checked over all rows.

| Table 7 row | Constraint | Scope | Quantity estimated from the data | Requirement |
|---|---|---|---|---|
| Monotonic | Eq. 3.9 — NOx vs SOI | SOI block | slope dNOx/dSOI (linear fit) | negative |
| Monotonic | Eq. 3.10 — PM vs λ | λ block | slope dPM/dλ | negative |
| U-shaped | Eq. 3.11 — HC vs λ | λ block | quadratic coefficient of HC(λ) | positive (convex) |
| Inverse | Eq. 3.12 — NOx–PM | SOI block | dNOx/dSOI × dPM/dSOI | negative (opposite directions) |
| Soft | Eq. 3.13 — η–NOx | SOI block | dη/dSOI × dNOx/dSOI | negative (opposite directions) |
| Non-neg | Eq. 3.14 — HC, NOx, CO2, PM ≥ 0 | all rows | minimum of each emission | non-negative |
| Bounds | 0 ≤ η ≤ 1 | all rows | minimum and maximum of η | within [0, 1] |

Three outputs are produced: the table (every row above, with a `detail`
column showing the individual slopes behind each product, the vertex of
the HC(λ) fit, and the per-emission minima), one scatter per gradient
relationship, and a direct view of the two trade-offs (PM vs NOx and η vs
NOx inside the SOI block, joined in increasing SOI order).

**How to read the results:**
- Values are in raw units; only the **sign** (or the bound) is compared.
- Eq. 3.12: the text states that the constraint "does not prescribe which
  direction corresponds to advancing versus retarding, only that the
  responses must be opposite". A disagreement here therefore does not
  depend on the SOI sign convention.
- Eq. 3.13: Sec. 3.3.2.2 says the constraint weight will be reduced or set
  to zero where the data shows no clear or a *positive* correlation. A
  disagreement here is anticipated by the text's own rule — provided the
  validity function that implements it (C1, Section 8) actually does so.
- Eq. 3.14 and the η bounds constrain the network's *predictions*; on the
  measured data they are only a sanity check.
- Not checkable with this OFAT design: the Sec. 3.3.1.1 statement that
  the NOx–SOI relationship stays monotonic across substitution rates from
  0% to 80%, because SOI is only swept at one substitution rate.

**Limits of this check:** it is a diagnostic, not a proof. The block
assignment is heuristic (see the caveat in A1); within a block the other
three inputs are not perfectly constant; the SOI block has few points;
and the quadratic coefficient for HC is sensitive to local irregularities
in the curve. A disagreement here is a reason to review the equation and
the SOI sign convention with the advisor before training the PINN — not
a conclusion on its own.

In [ ]:
# OFAT block inference -- identical to A1, Section 9
medians = {c: df[c].median() for c in INPUT_COLS}
ranges = {c: (df[c].max() - df[c].min()) for c in INPUT_COLS}
deviation = np.column_stack([
    np.abs(df[c].to_numpy() - medians[c]) / ranges[c] for c in INPUT_COLS
])
raw_labels = np.array(INPUT_COLS)[deviation.argmax(axis=1)]


def smooth_isolated_labels(labels, passes=2):
    out = list(labels)
    for _ in range(passes):
        changed = False
        for i in range(1, len(out) - 1):
            if out[i] != out[i - 1] and out[i - 1] == out[i + 1]:
                out[i] = out[i - 1]
                changed = True
        if not changed:
            break
    return np.array(out)


block_labels = smooth_isolated_labels(raw_labels)


def slope(y, x):
    return np.polyfit(x, y, 1)[0]


def sign_word(v):
    return "positive" if v > 0 else "negative"


# ---- gradient-based rows (Eq. 3.9-3.13), measured inside the OFAT block ----
# (name, block, kind, (output, input), second pair for trade-offs, sign required by Table 7)
TABLE7_GRADIENT_CHECKS = [
    ("Eq. 3.9 - NOx vs SOI",           "SOI",    "slope",   ("NOx", "SOI"),    None,           "negative"),
    ("Eq. 3.10 - PM vs lambda",        "lambda", "slope",   ("PM", "lambda"),  None,           "negative"),
    ("Eq. 3.11 - HC vs lambda",        "lambda", "quad",    ("HC", "lambda"),  None,           "positive"),
    ("Eq. 3.12 - NOx x PM (d/dSOI)",   "SOI",    "product", ("NOx", "SOI"),    ("PM", "SOI"),  "negative"),
    ("Eq. 3.13 - eta x NOx (d/dSOI)",  "SOI",    "product", ("eta", "SOI"),    ("NOx", "SOI"), "negative"),
]
QUANTITY_LABEL = {"slope": "slope (linear fit)",
                  "quad": "quadratic coefficient",
                  "product": "product of slopes"}

rows = []
for name, block, kind, (out_a, in_a), pair_b, expected in TABLE7_GRADIENT_CHECKS:
    mask = block_labels == block
    n_pts = int(mask.sum())
    assert n_pts >= 4, f"block {block} has only {n_pts} points -- too few to estimate a sign"
    y, x = df[out_a].to_numpy()[mask], df[in_a].to_numpy()[mask]
    if kind == "slope":
        value = slope(y, x)
        detail = f"r in block = {np.corrcoef(x, y)[0, 1]:+.2f}"
    elif kind == "quad":
        a2, a1, _ = np.polyfit(x, y, 2)
        value = a2
        vertex = -a1 / (2 * a2)
        where = "inside" if x.min() <= vertex <= x.max() else "outside"
        detail = (f"vertex at {in_a} = {vertex:.3f} ({where} the swept range "
                  f"{x.min():.3f}-{x.max():.3f})")
    else:
        out_b, in_b = pair_b
        s_a = slope(y, x)
        s_b = slope(df[out_b].to_numpy()[mask], df[in_b].to_numpy()[mask])
        value = s_a * s_b
        detail = f"d{out_a}/d{in_a} = {s_a:+.4g} ; d{out_b}/d{in_b} = {s_b:+.4g}"
    observed = sign_word(value)
    rows.append({
        "constraint": name, "scope": f"{block} block", "n_points": n_pts,
        "quantity": QUANTITY_LABEL[kind], "estimated_value": float(f"{value:.4g}"),
        "table7_requirement": expected, "observed": observed,
        "agrees_with_table7": observed == expected, "detail": detail,
    })

# ---- bounding rows (Eq. 3.14 and eta bounds), over all rows ----
EMISSIONS = ["HC", "NOx", "CO2", "PM"]
emission_mins = {c: float(df[c].min()) for c in EMISSIONS}
lowest = min(emission_mins.values())
rows.append({
    "constraint": "Eq. 3.14 - non-negativity (HC, NOx, CO2, PM)", "scope": "all rows",
    "n_points": df.shape[0], "quantity": "minimum over all emissions",
    "estimated_value": float(f"{lowest:.4g}"),
    "table7_requirement": "non-negative", "observed": "non-negative" if lowest >= 0 else "negative",
    "agrees_with_table7": lowest >= 0,
    "detail": " ; ".join(f"{c} min = {v:.4g}" for c, v in emission_mins.items()),
})

eta_raw = df["eta"].to_numpy()
eta_frac = eta_raw / 100 if eta_raw.max() > 1 else eta_raw   # A1 Section 7: stored as a fraction
in_bounds = bool((eta_frac >= 0).all() and (eta_frac <= 1).all())
rows.append({
    "constraint": "Bounds - 0 <= eta <= 1", "scope": "all rows",
    "n_points": df.shape[0], "quantity": "minimum and maximum of eta",
    "estimated_value": None,
    "table7_requirement": "within [0, 1]", "observed": "within [0, 1]" if in_bounds else "outside [0, 1]",
    "agrees_with_table7": in_bounds,
    "detail": f"eta min = {eta_frac.min():.4f} ; max = {eta_frac.max():.4f} (as a fraction)",
})

table7_checks = pl.DataFrame(rows)
flagged_table_html(table7_checks, "A2 -- Every Table 7 constraint checked against the data",
                    HTML_DIR / "A2_08_table7_signs_table.html",
                    flag_col="agrees_with_table7", is_flagged=lambda v: not v)
table7_checks

In [ ]:
# One scatter per relationship: rows of the relevant block in the output
# variable's color, all other rows in grey. Each panel title shows the
# correlation within the block and the global one, to make the dilution
# visible.
PANELS = [  # (output, input, block, fit degree)
    ("NOx", "SOI", "SOI", 1),
    ("PM", "SOI", "SOI", 1),
    ("eta", "SOI", "SOI", 1),
    ("PM", "lambda", "lambda", 1),
    ("HC", "lambda", "lambda", 2),
]

titles = []
for out, inp, block, _ in PANELS:
    mask = block_labels == block
    x_all, y_all = df[inp].to_numpy(), df[out].to_numpy()
    r_block = np.corrcoef(x_all[mask], y_all[mask])[0, 1]
    r_global = np.corrcoef(x_all, y_all)[0, 1]
    titles.append(f"{out} vs {inp}<br>r block={r_block:+.2f} | r global={r_global:+.2f}")

fig = make_subplots(rows=2, cols=3, subplot_titles=titles, vertical_spacing=0.18)
for k, (out, inp, block, degree) in enumerate(PANELS):
    r, c = divmod(k, 3)
    mask = block_labels == block
    x_all, y_all = df[inp].to_numpy(), df[out].to_numpy()
    fig.add_trace(go.Scatter(x=x_all[~mask], y=y_all[~mask], mode="markers",
                             marker=dict(color=SPLIT_COLORS["neutral"], size=6, opacity=0.5),
                             name="other rows", showlegend=(k == 0)),
                  row=r + 1, col=c + 1)
    fig.add_trace(go.Scatter(x=x_all[mask], y=y_all[mask], mode="markers",
                             marker=dict(color=VARIABLE_COLORS[out], size=9,
                                         line=dict(color=SPLIT_COLORS["reference"], width=0.5)),
                             name=f"{block} block", showlegend=False),
                  row=r + 1, col=c + 1)
    coefs = np.polyfit(x_all[mask], y_all[mask], degree)
    x_fit = np.linspace(x_all[mask].min(), x_all[mask].max(), 100)
    fig.add_trace(go.Scatter(x=x_fit, y=np.polyval(coefs, x_fit), mode="lines",
                             line=dict(color=VARIABLE_COLORS[out], width=2, dash="dash"),
                             showlegend=False),
                  row=r + 1, col=c + 1)
    fig.update_xaxes(title_text=inp, row=r + 1, col=c + 1)
    fig.update_yaxes(title_text=out, row=r + 1, col=c + 1)

fig.update_layout(height=720, width=1050,
                   title_text="Table 7 relationships within the OFAT block (colored) vs. other rows (grey)")
fig.show()
fig.write_html(str(HTML_DIR / "A2_08_table7_signs_scatter.html"), include_plotlyjs="inline")

In [ ]:
# Direct view of the two trade-off constraints (Eq. 3.12 and 3.13): inside
# the SOI block, one output plotted against the other, points joined in
# increasing SOI order. If the trade-off held, the path would slope DOWN
# (one output falls while the other rises).
soi_mask = block_labels == "SOI"
order = np.argsort(df["SOI"].to_numpy()[soi_mask])
soi_sorted = df["SOI"].to_numpy()[soi_mask][order]
SOI_SCALE = [[0, SPLIT_COLORS["neutral"]], [1, VARIABLE_COLORS["SOI"]]]

TRADEOFF_PANELS = [  # (x output, y output, title, what Table 7 expects)
    ("NOx", "PM", "Eq. 3.12 - PM vs NOx", "Table 7 expects PM to fall as NOx rises"),
    ("NOx", "eta", "Eq. 3.13 - eta vs NOx", "Table 7 expects eta to fall as NOx rises"),
]
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=[f"{t}<br><sup>{e}</sup>" for _, _, t, e in TRADEOFF_PANELS])
for k, (x_out, y_out, _, _) in enumerate(TRADEOFF_PANELS):
    x = df[x_out].to_numpy()[soi_mask][order]
    y = df[y_out].to_numpy()[soi_mask][order]
    fig.add_trace(go.Scatter(
        x=x, y=y, mode="lines+markers",
        line=dict(color=SPLIT_COLORS["neutral"], width=1),
        marker=dict(size=10, color=soi_sorted, colorscale=SOI_SCALE,
                    line=dict(color=SPLIT_COLORS["reference"], width=0.5),
                    showscale=(k == 0), colorbar=dict(title="SOI")),
        showlegend=False), row=1, col=k + 1)
    fig.add_annotation(x=x[0], y=y[0], text=f"lowest SOI ({soi_sorted[0]:.1f})",
                       showarrow=True, arrowhead=2, ax=40, ay=-30, row=1, col=k + 1)
    fig.add_annotation(x=x[-1], y=y[-1], text=f"highest SOI ({soi_sorted[-1]:.1f})",
                       showarrow=True, arrowhead=2, ax=-40, ay=30, row=1, col=k + 1)
    fig.update_xaxes(title_text=x_out, row=1, col=k + 1)
    fig.update_yaxes(title_text=y_out, row=1, col=k + 1)

fig.update_layout(height=480, width=1000,
                  title_text="Trade-off constraints inside the SOI block (points joined in increasing SOI order)")
fig.show()
fig.write_html(str(HTML_DIR / "A2_08_table7_tradeoffs_scatter.html"), include_plotlyjs="inline")

## Next

Stage A3 (normalisation + stratified train/val/test split) can reuse
`INPUT_COLS` / `OUTPUT_COLS` from this notebook. The VIF result above
(all ≈1, by design) means no input needs to be dropped or combined
before modelling.

The reference for checking the constraint signs learned by the PINN in
Phase C is the table in **Section 8** (`A2_08_table7_signs_table.html`),
not the global correlation matrices of Sections 4–5. Any constraint
flagged as disagreeing there needs to be resolved in the text and in C1
**before** training C4 — otherwise the physics loss pushes the model
away from what the data itself shows.